Rag stands for Reality-Augmented Generation
it takes pdfs and analyzes the data to give accurate answers.
\*No hallucination in AI, or madeup things.

Steps to learning RAG

1. Document loading and Splitting
   \*Chunking: dividing data in small parts. large enough to be meaningful and small enough to be in AIs capability to read.
   All chunks must be equal size
   Process
   we fetch arcticle from wikipedia and then make its little chunks in below code.


In [15]:
from bs4 import BeautifulSoup
import requests
import os
from dotenv import load_dotenv


def fetch_Article(url):
    header={"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
    response=requests.get(url, headers=header)
    soup=BeautifulSoup(response.text,"html.parser")
    paragraphs=soup.find_all("p")
    content=" ".join([p.get_text() for p in paragraphs])
    return content

def chunks(text,chunk_size=1000,overlap=100):
    print("Splitting text into chunks...")
    chunks=[]
    start=0
    while start<len(text):
        end=start+chunk_size
        chunk=text[start:end]
        chunks.append(chunk)
        start+=chunk_size-overlap
    print("Total chunks created: ",len(chunks))
    return chunks

if __name__=="__main__":
    url="https://en.wikipedia.org/wiki/Artificial_intelligence"
    text=fetch_Article(url)
    my_chunks=chunks(text)
    # print("1st chunk:",my_chunks[1])
    # print("2nd chunk:",my_chunks[2])
    # print("3rd chunk:",my_chunks[3])



Splitting text into chunks...
Total chunks created:  95


Text Embedding:
changing English words to numbers(vectors).
example:
The dog barked ->[0.12,-0.55,0.89]
The puppy barked ->[0.11,-0.54,0.90]
the car is fast ->[-0.88,-0.99,-0.12]

beacuse the first 2 are almost similar the numbers are close but the last one(car) is completely difirent so the num ers are far from them.
When a question is asked. it will be converted to numbers(vectors) and then we find answers using those numbers.


In [16]:
def create_embeddings(my_chunks):
    print("Sending ",len(my_chunks),"chunks to AI for embedding...")
    API_URL="https://api-inference.huggingface.co/pipeline/feature-extraction/sentence-transformers/all-MiniLM-L6-v2"
    api_key=os.getenv("Hugging_Face_API")
    headers={"Authorization":"Bearer "+api_key}
    